### Dataset ingestion and Domain normalization :

Loading a stratified 20,000 domains classifying into benign and DGA domains and isolated the Second level domain to prevent top level domain suffixes such as .com, .org, etc from manipulating character entropy and derived an output of sample domains of 5000 into sample_domains.csv

*Dataset sourced from [Prof. Christian Morbidoni's DGA_domains_dataset](https://github.com/chrmor/DGA_domains_dataset)*

In [5]:
!pip install -q shap lime xgboost scikit-learn pandas numpy matplotlib

import os
import time
import math
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
np.random.seed(42)

os.makedirs("figures", exist_ok=True)
os.makedirs("data", exist_ok=True)

def extract_sld(domain: str) -> str:
    clean = str(domain).lower().strip()
    parts = clean.split(".")
    if len(parts) >= 3 and parts[-2] in {"co", "com", "org", "net", "gov", "ac", "edu"}:
        return parts[-3]
    return parts[0]

DATASET_URL = "https://raw.githubusercontent.com/chrmor/DGA_domains_dataset/master/dga_domains_full.csv"
raw_df = pd.read_csv(DATASET_URL, header=None, names=["class_str", "family", "domain"]).dropna(subset=["domain"])
raw_df["label"] = (raw_df["class_str"].str.lower() == "dga").astype(int)

benign_df = raw_df[raw_df["label"] == 0].sample(n=10000, random_state=42)
dga_df = raw_df[raw_df["label"] == 1].sample(n=10000, random_state=42)

df = pd.concat([benign_df, dga_df], ignore_index=True)
df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)
df["sld"] = df["domain"].apply(extract_sld)
df = df[df["sld"].str.len() > 1].reset_index(drop=True)

df.head(5000)[["domain", "sld", "family", "label"]].to_csv("data/sample_domains.csv", index=False)

print(f"Total domains: {len(df):,}")
print(df["label"].value_counts().rename({0: "Benign (0)", 1: "DGA (1)"}))
print(df[["domain", "sld", "family", "label"]].head(8).to_string(index=False))

Total domains: 20,000
label
DGA (1)       10000
Benign (0)    10000
Name: count, dtype: int64
                        domain                        sld  family  label
         vqb1rrqev5hoxavfhs.cn         vqb1rrqev5hoxavfhs  rovnix      1
                      cuny.edu                       cuny   alexa      0
          spurscommunity.co.uk             spurscommunity   alexa      0
                  albelady.com                   albelady   alexa      0
          xdjmsdlbxwqbetwgf.tw          xdjmsdlbxwqbetwgf ranbyus      1
            pbraoqwiktcwxd.com             pbraoqwiktcwxd  ramnit      1
spiritreactcommitrulereact.com spiritreactcommitrulereact  matsnu      1
                  xhasters.com                   xhasters   alexa      0


### Lexical Feature extraction :

Extracting 6 character-level lexical metrics mathematically from each SLD to capture and train our engine the patterns to distinguish benign domains from DGA domains

In [6]:
VOWELS = set("aeiou")
CONSONANT_STREAK_RE = re.compile(r"[bcdfghjklmnpqrstvwxyz]+")

def shannon_entropy(s: str) -> float:
    n = len(s)
    if n <= 1:
        return 0.0
    counts = {}
    for ch in s:
        counts[ch] = counts.get(ch, 0) + 1
    return -sum((c / n) * math.log2(c / n) for c in counts.values())

def normalized_entropy(s: str) -> float:
    n = len(s)
    if n <= 1:
        return 0.0
    return shannon_entropy(s) / math.log2(n)

def vowel_ratio(s: str) -> float:
    if not s:
        return 0.0
    return sum(1 for ch in s if ch in VOWELS) / len(s)

def digit_ratio(s: str) -> float:
    if not s:
        return 0.0
    return sum(1 for ch in s if ch.isdigit()) / len(s)

def max_consonant_streak(s: str) -> int:
    streaks = CONSONANT_STREAK_RE.findall(s)
    return max((len(st) for st in streaks), default=0)

t0 = time.perf_counter()
df["sld_length"] = df["sld"].apply(len)
df["shannon_entropy"] = df["sld"].apply(shannon_entropy)
df["norm_entropy"] = df["sld"].apply(normalized_entropy)
df["vowel_ratio"] = df["sld"].apply(vowel_ratio)
df["digit_ratio"] = df["sld"].apply(digit_ratio)
df["max_consonant_streak"] = df["sld"].apply(max_consonant_streak)
lexical_time_ms = (time.perf_counter() - t0) * 1000

LEXICAL_COLS = [
    "sld_length",
    "shannon_entropy",
    "norm_entropy",
    "vowel_ratio",
    "digit_ratio",
    "max_consonant_streak"
]

print(f"Feature extraction completed in {lexical_time_ms:.2f} ms")
print(df.groupby("label")[LEXICAL_COLS].mean().round(4).to_string())

Feature extraction completed in 476.55 ms
       sld_length  shannon_entropy  norm_entropy  vowel_ratio  digit_ratio  max_consonant_streak
label                                                                                           
0         10.6491           2.8416        0.8724       0.3609       0.0207                2.4937
1         14.1494           3.1632        0.8587       0.2749       0.0366                4.7963
